# Experiment 4 trên Kaggle

Bật **Internet** trong Kaggle Notebook và dùng CPU. Notebook tải mã từ Graph-Robust-Planning trên GitHub rồi chạy pipeline của repo để đo robust regret tại root.

Bản đầy đủ dùng 4 planner, 8 mức mô phỏng từ 16 đến 2048 và 100 seed đánh giá. Mỗi cặp plan–exec tune riêng ngân sách TV `epsilon`. Frozen Lake và CliffWalking dùng plan 0.1, Gambler dùng plan 0.7. Cả ba dùng exec 0.4, 0.6 và 0.8.

Notebook tải mã nền đã có trên GitHub ở commit `c0c84ae`. Hai script Experiment 4 từ commit `aa9605c` được đính kèm trong notebook và áp dụng sau khi tải. Sửa `REPO_URL` nếu dùng fork có cùng commit nền. Với repo riêng tư, thêm Kaggle Secret tên `GITHUB_TOKEN` có quyền đọc repo.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Thangcoding/Graph-Robust-Planning.git"
GIT_REF = "c0c84ae46f0c11b2a51a8dcc333a2c39d3ffa6fb"
EXP4_REVISION = "aa9605cbf9b8a63571d0570c8993d228d053c43c"
PRIVATE_REPO = False
SMOKE_TEST = False
RUN_EXPERIMENT = True
ENVS = ["gambler", "frozen_lake", "cliffwalking"]
EXEC_THETAS = [0.4, 0.6, 0.8]
SIMS = [2**i for i in range(4, 12)]
WORKERS = len(os.sched_getaffinity(0))

# Thêm thư mục kết quả từ các lượt trước nếu muốn tiếp tục hoặc gộp nhiều lượt.
# Ví dụ Path("/kaggle/input/your-dataset/exp4-full-aa9605cbf9b8")
PREVIOUS_RESULTS = []

WORK = Path("/kaggle/working")
REPO = Path("/tmp") / f"graph-robust-planning-{GIT_REF[:12]}"
SCRIPT = "experiments/exp4_robust_regret.py"
PLOT = "experiments/results/experiement_4/plot_exp4.py"

## Tải mã và cài thư viện

Notebook chỉ checkout mã nguồn cần cho Experiment 4 để giảm dung lượng tải. Commit nền và phiên bản hai script được cố định để các lượt chạy và kết quả khôi phục dùng cùng một phiên bản.

In [ ]:
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if PRIVATE_REPO:
    from base64 import b64encode
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del token, auth

REPO.mkdir(parents=True, exist_ok=True)
def git(*args, **kwargs):
    return subprocess.run(["git", "-C", str(REPO), *args], check=True, env=git_env, text=True, **kwargs)

try:
    git("init", "--quiet")
    remotes = git("remote", capture_output=True).stdout.split()
    git("remote", "set-url" if "origin" in remotes else "add", "origin", REPO_URL)
    git("config", "remote.origin.promisor", "true")
    git("config", "remote.origin.partialclonefilter", "blob:none")
    git("sparse-checkout", "set", "--no-cone", "/src/", "/requirements.txt",
        "/experiments/*.py", f"/{PLOT}")
    git("fetch", "--depth=1", "--filter=blob:none", "origin", GIT_REF)
    git("checkout", "--detach", "FETCH_HEAD")
    revision = git("rev-parse", "HEAD", capture_output=True).stdout.strip()
except subprocess.CalledProcessError:
    raise RuntimeError("GitHub fetch/checkout thất bại. Kiểm tra Internet, commit và quyền đọc repo.") from None
finally:
    del git_env

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)

In [ ]:
# Hai script Experiment 4 từ commit aa9605c.
exp4_scripts = {
    'experiments/exp4_robust_regret.py': r'''"""Measure root robust regret under execution-model mismatch.

Each planner builds its tree at theta_plan. Its chosen root action is scored
by exact robust value iteration at theta_exec using the configuration's TV
budget. Tune epsilon, exploration constant and power separately for each
model pair, then evaluate the best configuration on independent seeds.
"""
from __future__ import annotations

import argparse
import json
import os
import sys
from multiprocessing import Pool

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import tune  # noqa: E402
from src.envs import Inventory  # noqa: E402
from value_iteration import solve  # noqa: E402

# at module level: the worker processes import this file again, they would not see a change made in main()
tune.ENVS["inventory"] = (Inventory, "lam")

ALGOS = ["robust-uct", "gs-robust-uct", "gsf-robust-uct", "gsf-robust-uct-plus"]
# Planning models match Exp3 (wall-clock time per search).
THETA_PLAN = {"frozen_lake": 0.1, "cliffwalking": 0.1, "gambler": 0.7, "inventory": 10.0}
SIMS = [2 ** exponent for exponent in range(4, 12)]
THETA_EXEC = [0.4, 0.6, 0.8]
EVAL_SEEDS = 100
EVAL_TAG = 15485863   # eval searches get their own random streams, apart from tune (TUNE_TAG) and run (RUN_TAG)
OPT_TOL = 1e-6        # robust regret below this counts as a robust-optimal action
OUT = os.path.join(tune.ROOT, "experiments", "results", "experiement_4")

TRUTH: dict = {}      # "robust" / "nominal" -> {state: Q*(state, .)}; set in every worker process


def init_worker(truth: dict) -> None:
    TRUTH.update(truth)


def root_states(robust_models: dict, n: int | None) -> list[int]:
    states = tune.scored_states(robust_models)
    if n is None or n >= len(states):
        return states
    return sorted(int(s) for s in np.random.default_rng(0).choice(states, size=n, replace=False))


# ---------------------------------------------------------------- tune (tune.py rounds, robust regret as the score)

def tune_robust(env, algo, sims, eps, states, workers, truth):
    configs = dict(enumerate(tune.make_configs(algo, sims, eps)))

    def score(configs, seeds, stage):
        searches = evaluate(env, algo, configs, states, seeds, workers, truth, stage, tune.TUNE_TAG)
        return searches.assign(regret_true=searches.robust_regret)

    screen = score(configs, tune.SCREEN_SEEDS, "screen")
    screened = tune.rank(tune.summarize(screen))
    finalists = screened[screened["rank"] <= tune.REFINE_TOP].config_id.tolist()
    refine = score({i: configs[i] for i in finalists}, tune.REFINE_SEEDS, "refine")
    pooled = tune.rank(tune.summarize(pd.concat([screen[screen.config_id.isin(finalists)], refine])))
    dropped = screened[~screened.config_id.isin(finalists)]
    results = pd.concat([pooled.assign(refined=True), dropped.assign(refined=False)], ignore_index=True)
    return pd.concat([screen, refine], ignore_index=True), results


def save_tune(out_dir: str, env: str, algo: str, sims: int, theta_exec: float, states: list[int], searches, results) -> None:
    os.makedirs(out_dir, exist_ok=True)
    searches.to_csv(os.path.join(out_dir, "searches.csv"), index=False)
    results.sort_values("rank").to_csv(os.path.join(out_dir, "results.csv"), index=False)
    summary = {"env": env, "algo": algo, "num_simulations": sims, "theta_plan": tune.THETAS[env][0],
               "theta_exec": theta_exec, "score": "robust regret at the root", "states": states, "top": tune.top_configs(results)}
    temporary = os.path.join(out_dir, "top3.json.tmp")
    with open(temporary, "w") as f:
        json.dump(summary, f, indent=2)
    os.replace(temporary, os.path.join(out_dir, "top3.json"))


# ---------------------------------------------------------------- eval

def search_once(job: dict) -> dict:
    seeds = np.random.SeedSequence([job["seed"], job["state"], round(job["theta_plan"] * 1e6), job["tag"]]).spawn(2)
    env_seed, planner_seed = (int(s.generate_state(1)[0]) for s in seeds)
    plan_env = tune.make_env(job["env"], job["theta_plan"], seed=env_seed)
    plan_env.reset(seed=env_seed)
    tune.set_state(plan_env, job["state"])
    planner = tune.make_planner(job["algo"], plan_env, **job["config"], seed=planner_seed)
    action = planner.plan(plan_env)
    config = job["config"]
    robust = TRUTH["robust"][config["uncertainty_budget"]]
    r_rob = tune.regret(robust, job["state"], action)
    return {**config, "config_id": job["config_id"], "round": job["round"],
            "theta_plan": job["theta_plan"], "theta_exec": TRUTH["theta_exec"],
            "seed": job["seed"], "state": job["state"], "action": action, "env_seed": env_seed,
            "planner_seed": planner_seed, "robust_regret": r_rob, "robust_opt": float(r_rob < OPT_TOL),
            "nominal_regret": tune.regret(TRUTH["nominal"], job["state"], action)}


def evaluate(env, algo, configs, states, seeds, workers, truth, stage, tag):
    jobs = [{"env": env, "algo": algo, "config": config, "config_id": config_id,
             "theta_plan": tune.THETAS[env][0], "state": state, "seed": seed, "round": stage, "tag": tag}
            for config_id, config in configs.items() for state in states for seed in seeds]
    print(f"[{stage}] {len(configs)} configs, {len(jobs)} searches", flush=True)
    rows = []
    with Pool(workers, initializer=init_worker, initargs=(truth,)) as pool:
        for done, row in enumerate(pool.imap_unordered(search_once, jobs, chunksize=4), 1):
            rows.append(row)
            if done % max(1, len(jobs) // 10) == 0:
                print(f"  {done}/{len(jobs)}", flush=True)
    return pd.DataFrame(rows).sort_values(["state", "seed"]).reset_index(drop=True)


def summarize(searches: pd.DataFrame, config: dict, env: str, algo: str) -> pd.DataFrame:
    """One row: per-state means averaged over the states; SE over the seeds of the per-seed state average."""
    metrics = ["robust_regret", "nominal_regret", "robust_opt"]
    per_state = searches.groupby("state")[metrics].mean().mean()
    per_seed = searches.groupby("seed")[metrics].mean()
    row = {"env": env, "algo": algo, "theta_plan": tune.THETAS[env][0], "theta_exec": searches.theta_exec.iat[0], **config,
           **{f"mean_{m}": per_state[m] for m in metrics},
           **{f"{m}_se": per_seed[m].sem() for m in metrics},
           "n_states": searches.state.nunique(), "n_seeds": searches.seed.nunique()}
    return pd.DataFrame([row])


# ---------------------------------------------------------------- main

def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--envs", nargs="+", default=["frozen_lake", "cliffwalking", "gambler"], choices=sorted(tune.ENVS))
    parser.add_argument("--algos", nargs="+", default=ALGOS, choices=ALGOS)
    parser.add_argument("--sims", nargs="+", type=int, default=SIMS)
    parser.add_argument("--eps", type=float, help="fix the TV budget instead of tuning it")
    parser.add_argument("--theta-exec", nargs="+", type=float, default=THETA_EXEC)
    parser.add_argument("--theta-plan", type=float, default=None, help="default: THETA_PLAN of the env")
    parser.add_argument("--states", type=int, default=None, help="fixed random sample of this many root states")
    parser.add_argument("--seeds", type=int, default=EVAL_SEEDS, help="eval seeds per state")
    parser.add_argument("--workers", type=int, default=os.cpu_count())
    parser.add_argument("--out", default=OUT)
    args = parser.parse_args()
    if args.seeds < 2:
        parser.error("--seeds must be at least 2 to estimate standard error")

    for env in args.envs:
        theta = THETA_PLAN[env] if args.theta_plan is None else args.theta_plan
        tune.THETAS[env] = [theta]
        budgets = tune.SEARCH_SPACE["uncertainty_budget"] if args.eps is None else [args.eps]
        for theta_exec in args.theta_exec:
            model = tune.make_env(env, theta_exec)
            robust = {eps: solve(model, tune.FIXED["gamma"], eps) for eps in budgets}
            truth = {"robust": robust, "nominal": solve(model, tune.FIXED["gamma"], 0.0),
                     "theta_exec": theta_exec}
            states = root_states(robust, args.states)
            if not states:
                raise ValueError(f"No scored root states for {env} at theta_exec={theta_exec:g}")
            pair = os.path.join(args.out, env, f"plan{theta:g}", f"exec{theta_exec:g}")
            print(f"######## {env}: plan={theta:g}, exec={theta_exec:g}, {len(states)} root states", flush=True)
            for sims in args.sims:
                for algo in args.algos:
                    print(f"================ {env} | {algo} | sims {sims} ================", flush=True)
                    tune_dir = tune.result_dir(os.path.join(pair, "tune"), algo, env, sims, args.eps)
                    tune_path = os.path.join(tune_dir, "top3.json")
                    if not os.path.exists(tune_path):
                        searches, results = tune_robust(env, algo, sims, args.eps, states, args.workers, truth)
                        save_tune(tune_dir, env, algo, sims, theta_exec, states, searches, results)
                    run_dir = tune.result_dir(os.path.join(pair, "run"), algo, env, sims, args.eps)
                    summary_path = os.path.join(run_dir, "summary.csv")
                    if os.path.exists(summary_path):
                        print("already run", run_dir, flush=True)
                        continue
                    with open(tune_path) as file:
                        config = next(entry for entry in json.load(file)["top"] if entry["rank"] == 1)["params"]
                    if algo == "gsf-robust-uct-plus":
                        config["max_depth"] = tune.PLANNERS[algo].config_cls(**config).max_depth
                    searches = evaluate(env, algo, {0: config}, states, range(args.seeds), args.workers,
                                        truth, "eval", EVAL_TAG)
                    summary = summarize(searches, config, env, algo)
                    os.makedirs(run_dir, exist_ok=True)
                    searches.to_csv(os.path.join(run_dir, "searches.csv"), index=False)
                    temporary = summary_path + ".tmp"
                    summary.to_csv(temporary, index=False)
                    os.replace(temporary, summary_path)
                    print(summary[["uncertainty_budget", "mean_robust_regret", "robust_regret_se"]]
                          .round(5).to_string(index=False), flush=True)
    print("exp4 done", flush=True)


if __name__ == "__main__":
    main()
''',
    'experiments/results/experiement_4/plot_exp4.py': r'''"""Plot one Experiment 4 figure per environment and execution parameter."""
import argparse
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
ENVS = {"frozen_lake": "FrozenLake", "cliffwalking": "CliffWalking", "gambler": "Gambler"}
PLANS = {"frozen_lake": 0.1, "cliffwalking": 0.1, "gambler": 0.7}
# same colors as the Exp1 / Exp2 plots
ALGOS = [("robust_uct", "Robust-UCT", "#2a78d6"),
         ("gs_robust_uct", "GS-Robust-UCT", "#eda100"),
         ("gs_robust_uct_f", "GSF-Robust-UCT", "#e87ba4"),
         ("gsf_robust_uct_plus", "GSF-Robust-UCT+", "#00836b")]
ROWS = [("robust_regret", "robust regret at the root"), ("robust_opt", "robust-optimal action rate")]
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e4e3de"

plt.rcParams.update({"font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False})


def plot(root, env, title, theta_exec):
    theta_plan = PLANS[env]
    pair = os.path.join(root, env, f"plan{theta_plan:g}", f"exec{theta_exec:g}")
    series = []
    for folder, label, color in ALGOS:
        pattern = os.path.join(pair, "run", folder, env, "sims*", "summary.csv")
        paths = glob.glob(pattern)
        if not paths:
            raise FileNotFoundError(f"No summaries for {label} in {pair}")
        results = pd.concat(map(pd.read_csv, paths)).sort_values("num_simulations")
        for metric, _ in ROWS:
            if results[f"{metric}_se"].isna().any() or results.n_seeds.lt(2).any():
                raise ValueError(f"Standard error requires at least two evaluation seeds in {pattern}")
        series.append((label, color, results))

    ticks = series[0][2].num_simulations.tolist()
    if any(results.num_simulations.tolist() != ticks for _, _, results in series):
        raise ValueError(f"Simulation budgets differ across planners in {pair}")

    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, layout="constrained")
    for ax, (metric, ylabel) in zip(axes, ROWS):
        for label, color, results in series:
            x, y, se = results.num_simulations, results[f"mean_{metric}"], results[f"{metric}_se"]
            ax.fill_between(x, y - se, y + se, color=color, alpha=0.22, linewidth=0)
            ax.errorbar(x, y, yerr=se, color=color, linewidth=1.8, elinewidth=0.9,
                        capsize=2, marker="o", markersize=4, label=label)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
    axes[0].legend(loc="best", ncols=2, frameon=False)
    axes[-1].set_xscale("log", base=2)
    axes[-1].set_xticks(ticks, labels=[str(n) for n in ticks])
    axes[-1].minorticks_off()
    axes[-1].set_xlabel("Simulations per search")
    fig.suptitle(f"{title} — θ_plan = {theta_plan:g}, θ_exec = {theta_exec:g} (±1 SE)", color=INK)
    output = os.path.join(root, f"exp4_robust_regret_{env}_plan{theta_plan:g}_exec{theta_exec:g}")
    for ext in ("png", "pdf"):
        fig.savefig(f"{output}.{ext}", dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("saved", f"{output}.png")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--results", default=HERE)
    args = parser.parse_args()
    for env, title in ENVS.items():
        for theta_exec in (0.4, 0.6, 0.8):
            plot(args.results, env, title, theta_exec)
''',
}
for relative_path, source in exp4_scripts.items():
    destination = REPO / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")

## Chạy hoặc tiếp tục thí nghiệm

Có thể chia theo `ENVS`, `EXEC_THETAS` hoặc `SIMS` để chạy qua nhiều phiên. Lưu output của từng phiên rồi thêm các thư mục kết quả vào `PREVIOUS_RESULTS` trong phiên tiếp theo. Dùng `RUN_EXPERIMENT = False` nếu chỉ muốn gộp dữ liệu đã chạy và vẽ hình. Chế độ thử dùng một root state, hai seed và \(N=16\).

In [ ]:
mode = "smoke" if SMOKE_TEST else "full"
RESULTS = WORK / f"exp4-{mode}-{EXP4_REVISION[:12]}"
RESULTS.mkdir(exist_ok=True)
for source in PREVIOUS_RESULTS:
    if source.name != RESULTS.name or not source.is_dir():
        raise ValueError(f"Cần thư mục kết quả tên {RESULTS.name}, nhận được {source}")
    for path in source.glob("*/plan*/exec*/**/*"):
        if not path.is_file() or path.suffix not in {".csv", ".json"}:
            continue
        target = RESULTS / path.relative_to(source)
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

runtime_env = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1",
               "MKL_NUM_THREADS": "1", "MPLBACKEND": "Agg"}
if RUN_EXPERIMENT:
    budgets = [16] if SMOKE_TEST else SIMS
    seed_args = ["--states", "1", "--seeds", "2"] if SMOKE_TEST else ["--seeds", "100"]
    command = [sys.executable, "-u", SCRIPT, "--envs", *ENVS,
               "--theta-exec", *map(str, EXEC_THETAS), "--sims", *map(str, budgets),
               "--workers", str(WORKERS), "--out", str(RESULTS), *seed_args]
    subprocess.run(command, cwd=REPO, env=runtime_env, check=True)
print(RESULTS)

## Kiểm tra tiến độ và vẽ 9 hình

Khi đủ dữ liệu, notebook xuất một hình cho mỗi cặp plan–exec dưới dạng PNG và PDF. Dải và thanh sai số thể hiện ±1 standard error trên trung bình theo seed.

In [ ]:
from IPython.display import Image, display

algorithms = ("robust_uct", "gs_robust_uct", "gs_robust_uct_f", "gsf_robust_uct_plus")
plans = {"frozen_lake": 0.1, "cliffwalking": 0.1, "gambler": 0.7}
exec_grid = (0.4, 0.6, 0.8)
expected_sims = [16] if SMOKE_TEST else [2**i for i in range(4, 12)]
complete_pairs = 0
for env, plan in plans.items():
    for theta_exec in exec_grid:
        run = RESULTS / env / f"plan{plan:g}" / f"exec{theta_exec:g}" / "run"
        paths = [run / algo / env / f"sims{n}" / "summary.csv"
                 for algo in algorithms for n in expected_sims]
        completed = sum(path.is_file() for path in paths)
        complete_pairs += completed == len(paths)
        print(f"{env:14} plan={plan:g} exec={theta_exec:g}   {completed}/{len(paths)}")

if complete_pairs == 9:
    subprocess.run([sys.executable, PLOT, "--results", str(RESULTS)],
                   cwd=REPO, env=runtime_env, check=True)
    for image in sorted(RESULTS.glob("exp4_robust_regret_*_plan*_exec*.png")):
        display(Image(filename=str(image), width=720))
else:
    print(f"Đã hoàn thành {complete_pairs}/9 cặp. Kết quả từng cấu hình nằm trong {RESULTS}.")